# Running [passim](https://github.com/dasmiq/passim) on a small corpus

This notebook runs passim on a set of texts. It is intended to be a no-code interface for running passim (you just need to upload files into colab and press the play buttons on each code block).

Feel free to make a copy of this notebook and adapt it for your needs (if you have some knowledge of Python and the Command Line Interface).

## Step 1. Uploading your texts

To start the script, upload your texts into this notebook, either in the root folder or in a new folder. In the following code blocks, the script will ask you  for the input texts: give it the path to the folder you uploaded.

Some remarks on the file names of your text files:
1. Make sure you name the text files with proper, **unique book ids** so that you can identify those books' ids in the passim text-reuse data.
2. We suggest to give all the filenames the **extension ".txt"**, as the script takes whats comes before ".txt" as the book IDs and uses these IDs later in output folder and file names.
3. For file names, use a combinaton of letters and numbers and **avoid special characters, whitespaces, and any unconventional letters** as much as possible.

## Step 2: Preprocessing

First, run the code block below to install the openiti Python library.

In [ ]:
!pip install openiti

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.1/270.1 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.7/432.7 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.4 MB/s eta 0:00:00
  Created wheel for pypyodbc: filename=pypyodbc-1.3.6-py3-none-any.whl size=22857 sha256=4f04b24c4676dc52284911824dd1146d50eba0197e9424b96739be0ceb3dcc7c
  Stored in directory: /root/.cache/pip/wheels/cc/0f/d7/04afb4d86f85c4969168e5a366dbf8c17751159188c2c677ed
Successfully built pypyodbc


### Step 2.1: Chunking the text

In order to make comparing books of various length easier, we divide the books into equal-lenght chunks - the KITAB project uses chunks of 300 tokens.

The script in the code block below inserts "milestone" tags into the original text files that mark the end of each chunk, so that it becomes possible to trace a chunk to its position in the text.

An example from OpenITI of a milestone marker (ms173) in a text - this tag means that after this point, the 174th 300-word chunk of text starts:

<blockquote>
لو انه أغنى بكيت كخندف % على الياس حتى ملها السر تندب
 إذا مؤنس لاحت خراطيم شمسه % بكت غدوة حتى ترى الشمس تغرب
 يعني بقوله مؤنس يوم الخميس لأن العرب كانت تسمي الأيام بغير أسمائها في
هذا الوقت فكانت تسمي الأحد الأول والاثنين أهون والثلاثاء جبار والاربعاء
دبار والخميس مؤنسا والجمعة عروبة والسبت شيار وكانوا يسمون أيام الشهر <mark style ="background-color:#0000FF">ms173</mark>
عشرة أسماء كل ثلاث ليال اسم فالثلاث التي أول الهلال الغرر ثم النفل ثم
التسع ثم العشر ثم البيض ثم الظلم ثم الخنس ثم الحنادس ثم المحاق والآخر
ليلة السرار إذا استسر الهلال وكانوا يسمون المحرم مؤتمرا وصفرا ناجرا
وربيعا الأول خوان وربيعا الآخر وبصان وجمادى الأول حنين وجمادى الآخرة ربى
ورجبا الأصم وشعبان العاذل ورمضان ناتقا وشوالا وعلا وذا القعدة ورنة وذا
الحجة بركا وكان آخرون من العرب يسمون الثلاث ليال من أول الشهر هلالا ثم
ثلاث قمر حين يقمر ثم ثلاث بهر حين يضيء ويبهر لونه وثلاث نقل وثلاث بيض
وثلاث درع وثلاث ظلم وثلاث حنادس وثلاث دآدي وليلتان محاق وليلة سرار
وولد لطانجة بن إلياس اد فتفرقت من ولد اد بن طانجة أربع
</blockquote>

Instructions:
1. Run the script by pressing the button
2. It will ask you for the path to the input files. Please provide the name of the folder you uploaded in step 1, and press Enter.
3. Next, it will ask you to specify the length of the chunks (milestones). The KITAB project uses 300-word chunks, but if you have a good reason, you can use another number. Press Enter after providing the number, and the script will add milestone tags into your texts.

NB: pointers to adapt the script if needed:
- The script is designed to split off a metadata header from the body of the text. By default, it uses the pattern that ends the metadata header of OpenITI texts ("#META#Header#End") to do this; if your text contains a different pattern to split the header from the body of the text, change the `SPLITTER` constant at the top of the script. The script also works if the text does not have any splitter and takes the whole content of the text file as the body text.
- The default pattern for milestone tags is "\s{1}ms[A-Z]?\d+" (a space followed by the letters "ms", optionally an upper-case letter, and one or more digits). For a different pattern, change the constant value `MS_REMOVE_REGEX`.

In [ ]:
# Masoumeh's passim input script

#!/usr/bin/env python3
"""
Insert fixed-length milestones into text files based on Arabic token counts.
"""

import os
import re
import sys
import math
from collections import Counter
from itertools import groupby
from typing import Optional
import openiti.helper.ara as ara

# Constants
SPLITTER = "#META#Header#End#"
FILE_PATTERN = re.compile(r".*(\.txt)$")
MS_FIND_REGEX = re.compile(r"ms[A-Z]?\d+")
MS_REMOVE_REGEX = re.compile(r"\s{1}ms[A-Z]?\d+")
MILESTONE300_REGEX = re.compile(r"\s*Milestone300")
# Set to True to process files even when the SPLITTER is missing
PROCESS_WITHOUT_SPLITTER = True


def insert_milestones(
    filepath: str, length: int, last_ms: int, log_file
) -> Optional[int]:
    """
    Read `filepath`, strip old milestone tags, and insert new ones every `length`
    Arabic tokens, continuing count from `last_ms`. Returns final ms count,
    or None on error.
    """
    # filename = os.path.basename(filepath)
    # name_base = re.split(r"-[a-z]{3}\d", filename)[0]

    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()

    # Split header/body, optionally skipping the missing‐splitter error
    if SPLITTER in content:
        header, body = map(str.rstrip, content.split(SPLITTER, 1))
        log_file.write(f"[DONE] Splitter in {filepath}, milestoned!\n")
    elif not PROCESS_WITHOUT_SPLITTER:
        log_file.write(f"[ERROR] Missing splitter in {filepath}, no changes made!\n")
        return None
    else:
        # No splitter but we're configured to proceed:
        header = ""
        body = content#.rstrip()
        log_file.write(f"[WARN] Missing splitter in {filepath}, milestoned!\n")

    # Remove old milestone tags, including the old pattern (Milestone300)
    body = MS_REMOVE_REGEX.sub("", body)
    body = MILESTONE300_REGEX.sub("", body)

    # Check for stray IDs
    # remaining = MS_FIND_REGEX.findall(body)
    # if len([m for m in remaining if m != ""]) > 1:
    #     log_file.write(f"[ERROR] Remaining IDs in {filepath}: {remaining}\n")
    #     return None

    # Count Arabic tokens
    total_ar = ara.ar_tok_cnt(body)
    pad_len = len(str(math.floor(total_ar / length)))
    tokens = re.findall(r"\w+|\W+", body)

    count, ms_count = 0, last_ms
    new_parts = []

    for i, tok in enumerate(tokens):
        new_parts.append(tok)
        if re.search(ara.ar_tok, tok):
            count += 1

        # Time to insert a milestone?
        if count >= length or i == len(tokens) - 1:
            ms_count += 1
            tag = f" ms{ms_count:0{pad_len}d}"
            new_parts.append(tag)
            count = 0

    new_body = "".join(new_parts)

    # Verify no corruption
    if MS_REMOVE_REGEX.sub("", new_body) != body:
        log_file.write(f"[ERROR] Content mismatch in {filepath}\n")
        return None

    # Check for duplicates
    ids = MS_FIND_REGEX.findall(new_body)
    dupes = {k: v for k, v in Counter(ids).items() if v > 1}
    if dupes:
        log_file.write(f"[ERROR] Duplicate IDs in {filepath}: {dupes}\n")
        return None

    # Write out
    if SPLITTER in content:
      final = header + "\n\n" + SPLITTER + "\n\n" + new_body
    else:
      final = new_body
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(final)

    return ms_count


def process_files(root_folder: str, ms_length: int) -> None:
    """
    Walk `root_folder`, find matching text files, and insert milestones.
    """
    if not os.path.isdir(root_folder):
        print(f"Invalid path: {root_folder}")
        sys.exit(1)

    log_path = os.path.join(root_folder, "../milestone_log.txt")
    with open(log_path, "w", encoding="utf-8") as log_file:
        for root, _, files in os.walk(root_folder):
            # Filter book files
            book_files = [
                f for f in files if FILE_PATTERN.match(f)
            ]
            for fname in book_files:
                insert_milestones(os.path.join(root, fname), ms_length, 0, log_file)

# def main():
folder = input("Enter the path to the input folder: ").strip()
try:
    length = int(input("Enter the length of milestones: ").strip())
except ValueError:
    print("Milestone length must be an integer.")
    sys.exit(1)

process_files(folder, length)
print("Done!")

Enter the path to the input folder: data
Enter the length of milestones: 300
Done!


### Step 2.2: Generating input files for passim

Run the script in the code block below, which generates input files for passim from the chunked input texts. The script will ask you again for the path to the folder where you have uploaded the texts. Provide the path and press Enter to start the process.

The passim input files are in [JSON Lines format](https://jsonlines.org//) : they contain containing one JSON record per line. Each of these records represents a single text chunk and look like this:

```
{"id": "d1", "series": "abc", "text": "This is text."}
```

("id" being the milestone number, and "series" the text filename)

In addition to the above fields, other metadata included in the record for each document will be passed through into the output.


In [ ]:
# prepare passim inputs
import os
import re
import sys
import csv
from itertools import groupby

import pandas as pd
from openiti.helper import funcs, ara


def mechanical_chunking(filename, text, output_dir, milestone_pattern, chunk_size):
    """
    Split `text` on each occurrence of `milestone_pattern`, clean it,
    and write JSON-like chunks of size `chunk_size` to files in `output_dir`.
    Returns 1 if any chunks were written, 0 if the last expected chunk already exists.
    """
    # Derive a simple book ID and prepare output path prefix
    book_id = re.sub(r"(\.txt)$", "", filename)
    print(book_id)
    prefix = os.path.join(output_dir, book_id)

    # If the final chunk file already exists, skip
    final_path = f"{prefix}-{chunk_size:05d}"
    if os.path.exists(final_path):
        return 0

    # Split on the milestone markers, keeping them in the list
    parts = re.split(fr"({milestone_pattern})", text)
    template = '{{"id":"{id}", "series":"{series}", "text":"{text}", "seq":{seq}}}'
    records = []
    counter = 0
    i = 0

    while i < len(parts) - 2:
        counter += 1
        marker = parts[i + 1]
        # remove vowels
        clean_text = ara.denoise(parts[i])
        # clean text
        clean_text = funcs.text_cleaner(clean_text)

        seq = int(re.sub(r"\D", "", marker))
        rec = template.format(
            id=f"{book_id}.{marker}",
            series=book_id,
            text=clean_text,
            seq=seq
        )
        records.append(rec)

        # Write out in batches of chunk_size
        if counter % chunk_size == 0:
            path = f"{prefix}-{counter:05d}.json"
            with open(path, "w", encoding="utf8") as fw:
                fw.write("\n".join(records))
            records = []

        i += 2

    # Write any remaining records
    # Round up to the next multiple of chunk_size
    final_counter = ((counter + chunk_size - 1) // chunk_size) * chunk_size
    path = f"{prefix}-{final_counter:05d}.json"
    with open(path, "w", encoding="utf8") as fw:
        fw.write("\n".join(records))

    return 1


main_folder = input("Enter the path to the text folder: ").strip()
target_folder = "passim_inputs" #input("Enter the path to write the new files: ").strip()
os.makedirs(target_folder, exist_ok=True)

# Validate paths
if not os.path.exists(main_folder):
  print(main_folder)
  print(f"Invalid path: {main_folder}", file=sys.stderr)
  sys.exit(1)

print("\nGenerating mechanical passim corpus...\n")

# Constants
milestone = r"ms[A-Z]?\d+"
SPLITTER = "#META#Header#End#"
chunk_size = 1000
written = []
total = 0

for root, _, files in os.walk(main_folder):
    # Match files like filename.txt
    pattern = r".+(\.txt)$"
    candidates = [f for f in files if re.match(pattern, f)]
    if not candidates:
        continue

    # Process each file
    for filename in candidates:
        path = os.path.join(root, filename)
        with open(path, encoding="utf8") as file:
            content = file.read()
            if SPLITTER in content:
              body = content.split(SPLITTER)[1]
            else:
              body = content

        status = mechanical_chunking(
            filename, body, target_folder, milestone, chunk_size
        )
        total += status
        book_id = re.sub(r"(\.txt)$", "", filename)
        written.append((book_id, status))

        if total % 100 == 0:
            print(f"\nProcessed: {total}\n" + "=" * 20 + "\n")

# Write a log of what was written
with open("corpus_log.csv", "w", newline="", encoding="utf8") as log_f:
    writer = csv.writer(log_f)
    writer.writerow(["book_id", "write_status"])
    writer.writerows(written)

print("Done! Your input files for passim are now in the", target_folder, "folder")


Enter the path to the text folder: data

Generating mechanical passim corpus...

0571IbnCasakir.TarikhDimashq.JK000916-ara1
0463KhatibBaghdadi.TarikhBaghdad.Shamela0000736-ara2
Done! Your input files for passim are now in the passim_inputs folder


## Step 3: Run passim

Run the code block below to download passim using python's pip installer.

In [ ]:
# Install passim

!pip install git+https://github.com/dasmiq/passim.git

  Cloning https://github.com/dasmiq/passim.git to /tmp/pip-req-build-ta63tqxj
  Running command git clone --filter=blob:none --quiet https://github.com/dasmiq/passim.git /tmp/pip-req-build-ta63tqxj
  Resolved https://github.com/dasmiq/passim.git to commit e4eea1c89c4182094d119047c85032cd9163ea17
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 6.6 MB/s eta 0:00:00
  Created wheel for passim: filename=passim-2.0.0-py3-none-any.whl size=15556 sha256=8bf2de2af366f6532a37ee33f0b27c919c615ddc2bc913bce0b65a36a7b11c1c
  Stored in directory: /tmp/pip-ephem-wheel-cache-8oxm28bc/wheels/b0/51/a2/50fb4f12befcc05270d5ad947734813b1e14df43e9f21862ad
Successfully built passim


Then, run the below code block to run passim,

*   List item
*   List item

invoking it through SPARK.

Passim has many parameters that can be tweaked. KITAB uses the following parameters:

```
--pairwise --filterpairs 'gid < gid2'
```

* `--pairwise` invokes passim to output pairwise alignments between all matching passages (in addition to the cluster data)
* `--filterpairs 'gid < gid2'` tells passim not to compare the same pair of chunks twice (chunk 1 to chunk 2 and chunk 2 to chunk 1)

For full documentation of passim and how to run, see https://github.com/dasmiq/passim.

NB: This script is designed to run cleanly, cleaning up any previous passim outputs in your file system before running passim.

In [ ]:
# FROM DAVID'S NOTEBOOK: https://colab.research.google.com/github/dasmiq/passim/blob/main/docs/passim_quickstart.ipynb#scrollTo=hZBefy-B1S01
# This takes input files in passim_inputs dir (created by the above steps)

# Delete old output
#!rm -r passim_output_json/
import shutil
if os.path.exists('passim_output_json'):
  shutil.rmtree('passim_output_json')

# Run passim:
!SPARK_SUBMIT_ARGS="--driver-memory 8G --executor-memory 8G" passim --pairwise --filterpairs 'gid < gid2' passim_inputs passim_output_json >& out_cluster.err

print("Done. The output files are in the passim_output_json folder")

Once the script has finished, passim's output files will be located in the folder `passim_output_json`.

## Step 4: Post-processing

### 4.1 Convert the output files to pairwise csvs compatible with KITAB apps

The code block below takes the output from passim (which is in json format) and creates from it a [tab-separated values file (tsv) ](https://en.wikipedia.org/wiki/Tab-separated_values) for each pair of books for which passim found chunks that contained text reuse. These tsv files can be loaded into the KITAB apps.

The columns include:

* begin: begin point of alignment in characters
* begin2: begin point of alignment2 in characters
* end: end point of alignment in characters
* end2: end point of alignment2 in characters
* gid: hash of series field
* gid2: hash of series2 field
* id: full id of text chunk, including book id and milestone id in book 1
* id2: full id of text chunk 2, including book id and milestone id in book 2
* matches: number of matches in aligned sequences
* s1: aligned sequence in book 1
* s2: aligned sequence in book 2
* seq: chunk (milestone) id in book 1. This value has been included as metadata in the record for each document in passim input in the preparation process and then is passed through into the output.
* seq2: chunk (milestone) id in book 2. This value has been included as metadata in the record for each document in passim input in the preparation process and then is passed through into the output.
* series: series identifier
* series2: series identifier for text 2
* uid: hash of document id in book 1
* uid2: hash of document id in book 2

Following columns are added by the post-processing script:

* w_match: count of words matched in the alignment (excluding whitespace)
* ch_match: count of characters matched in the alignment
* align_len: length of the alignment in s1
* matches_percentage: percentage of words match in s1


In [ ]:
# Masoumeh's script for outputs

# It will take 'passim_output_json/align.json' as input

#!/usr/bin/env python3
import pandas as pd

"""
This script augments Passim outputs (JSON or Parquet) with:
  - ch_match: character matches excluding whitespace
  - align_len: length of the aligned string
  - matches_percentage: (matches / align_len) * 100
  - w_match: word matches at identical positions

Outputs are written as partitioned CSVs by series/series2,
and post-processed to remove hidden files and rename parts.
"""

import os
import re
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.types import IntegerType, FloatType
import pyspark.sql.functions as F

# ──────────────────────────────────────────────────────────────────────────────
# CONFIGURE YOUR INPUT & OUTPUT PATHS
# ──────────────────────────────────────────────────────────────────────────────
INPUT_PATH = "passim_output_json/align.json"
OUTPUT_PATH = "passim_output_csv/pairwise-alignments"  # directory where CSVs will be written
# ──────────────────────────────────────────────────────────────────────────────


def word_count(s1: str, s2: str) -> int:
    """Count matching words between two aligned strings at identical positions."""
    cnt = 0
    i, n = 0, len(s1)
    while i < n:
        start = i
        while i < n and s1[i] != " ":
            i += 1
        if s1[start:i] == s2[start:i]:
            cnt += 1
        i += 1  # skip the space
    return cnt


def ch_count(s1: str, s2: str) -> int:
    """Count matching non-whitespace characters at identical positions."""
    return sum(1 for a, b in zip(s1, s2) if a == b and a != " ")


def match_percentage(matches: int, length: int) -> float:
    """Compute match percentage; return 0.0 if length is zero."""
    return (matches / length) * 100 if length > 0 else 0.0


def process_alignment(input_path: str, output_path: str) -> None:
    spark = SparkSession.builder.appName("Stats").getOrCreate()

    # Register UDFs
    word_match_udf = F.udf(word_count, IntegerType())
    ch_match_udf = F.udf(ch_count, IntegerType())
    align_len_udf = F.udf(lambda s: len(s), IntegerType())
    percent_udf = F.udf(match_percentage, FloatType())

    # Determine format from extension
    fmt = Path(input_path).suffix.lower()
    if fmt == ".json":
        src_format = "json"
    elif fmt == ".parquet":
        src_format = "parquet"
    else:
        raise ValueError(f"Unsupported input format: {fmt}")

    # Load, dedupe, and drop nulls
    df = (
        spark.read
             .format(src_format)
             .option("encoding", "UTF-8")
             .load(input_path)
             .distinct()
             .na.drop()
    )
    # new_col_names = {'id':'id1', 'first': 'first1', 'uid':'uid1', 'bw':'bw1',
    #                 'ew': 'ew1', 'begin':'b1', 'begin2':'b2', 'end':'e1',
    #                 'end2': 'e2', 'len': 'len1',
    #                 'tok': 'tok1', 'seq': 'seq1', 'gid': 'gid1'
    #                 , 'series': 'series1', 'mathces_percentage': 'matches_percent'}
    #                 #  , 'series2': 'series_b2'}

    # for old_name, new_name in new_col_names.items():
    #     df = df.withColumnRenamed(old_name, new_name)

    # Compute new columns
    df2 = (
        df
        .withColumn("ch_match", ch_match_udf("s1", "s2"))
        .withColumn("align_len", align_len_udf("s1"))
        .withColumn(
            "matches_percentage",
            F.when(F.col("align_len") == 0, F.lit(0.0))
             .otherwise(percent_udf(F.col("matches"), F.col("align_len")))
        )
        .withColumn("w_match", word_match_udf("s1", "s2"))
        .withColumn('series_b1', F.col('series')) \
        .withColumn('series_b2', F.col('series2')) \

    )

    # Write partitioned CSV output
    df2.repartition("series", "series2") \
       .sortWithinPartitions("id", "id2") \
       .write \
       .partitionBy("series", "series2") \
       .format("csv") \
       .option("header", "true") \
       .option("delimiter", "\t") \
       .mode("overwrite") \
       .save(output_path)

    spark.stop()



    # Post-process: remove hidden files, rename part files, and clean directories
    for root, _, files in os.walk(output_path, topdown=False):
        root_path = Path(root)

        # Inside a series2 partition
        if any(p.startswith("series2=") for p in root_path.parts):
            # Delete hidden files (e.g. ._SUCCESS)
            for f in files:
                if f.startswith("."):
                    (root_path / f).unlink()

            # Rename the first part file to SERIES_SERIES2.csv
            parts = [f for f in files if f.startswith("part")]
            if parts:
                series = next((p.split("=", 1)[1] for p in root_path.parts if p.startswith("series=")), "")
                series2 = next((p.split("=", 1)[1] for p in root_path.parts if p.startswith("series2=")), "")
                new_name = f"{series}_{series2}.csv"
                parent = root_path.parent
                os.rename(root_path / parts[0], parent / new_name)

            # Remove the now-empty series2 directory
            root_path.rmdir()

        # Rename the series directory to drop the prefix
        elif any(p.startswith("series=") for p in root_path.parts):
            new_dir = re.sub(r"series=", "", str(root_path))
            os.rename(root, new_dir)


# if __name__ == "__main__":
if os.path.exists(OUTPUT_PATH):
  shutil.rmtree(OUTPUT_PATH)
os.makedirs(OUTPUT_PATH)
#print(INPUT_PATH)
if not os.path.exists(INPUT_PATH):
  print(f"Invalid path: {INPUT_PATH}", file=sys.stderr)
  sys.exit(1)
process_alignment(INPUT_PATH, OUTPUT_PATH)

print("Done! The output can be found in the passim_output_csv/pairwise-alignments folder")



Done! The output can be found in the passim_output_csv/pairwise-alignments folder


### 4.2 Combine all data text reuse data per text ("one-to-all")


The code block below takes the pairwise tsv files generated by the first post-processing step and creates for every "target" text, two tsv files that contains metadata about the text reuse detected with this target text in all other texts in the corpus. These tsv files can be loaded into the KITAB apps for visualization.

1. alignment data:

The first file (`<text_id>_all.csv`) contains the data of all alignments with this text in the corpus:

The columns include:

* ms1: the milestone in the target text
* b1: begin point of alignment in characters in the target text
* e1: end point of alignment in characters in the target text
* id2: text ID of the other text
* ms1: the milestone in the other text
* b1: begin point of alignment in characters in the other text
* e1: end point of alignment in characters in the other text
* ch_match: count of characters matched in the alignment
* matches_percent: percentage of words in the target text matched in the other text

2. statistics:

The second file (`<text_id>_stats.csv`) contains statistics and metadata for every book in the corpus for which passim detected alignments with the target text:

The columns include:
* id: text ID of the other text
* book: book URI of the other text ()
* alignments: number of alignments with the target text
* ch_match: total count of characters matched in alignments with the target text

In [ ]:
#TODO

## Step 5: Visualisation and analysis

After the script has run, you will find the outputs in the 'passim_output_csv/pairwise-alignments' folder. You can download an output file and read it directly in **excel or google sheets** or upload it into one of the KITAB apps.

### 5.1 Read the files in a spreadsheet program

You can download and read an output tsv file in **excel or google sheets**.

NB: although the output files have the extension ".csv", they use tabs as separators. To open them in any spreadsheet software, make sure to use `Tab` as the separator, not comma!

As the output is a tsv and contains Arabic script, to open it in excel you will need to import it through the data tab. To do so follow these steps:
1. Open Excel and create a new spreadsheet
1. Go to the 'Data' tab
1. Click 'Get Data' --> 'From File' --> 'From Text/CSV'
1. Select the passim output tsv file that you have downloaded from colab and click 'import'
1. A modal will appear (65001:Unicode (UTF-8) should appear in the top left dropdown) - click 'Load'
1. Your data will load into excel

For larger outputs (greater than 300 rows) we recommend uploading the tsv file to google drive and viewing it using google sheets (this will be more stable).


### 5.2 Visualize pairwise relations

You can upload the tsv file to the **[KITAB visualisation app](https://kitab-project.org/explore/#/visualise/)** (choose "Upload a tsv file") to get a clear overview of the text reuse detected between a pair of texts.

More info on this visualisation can be found [here](https://kitab-project.org/data/viz#the-pairwise-text-reuse-visualisation).

### 5.3 Explore parallel passages in the KITAB diffViewer

Alternatively, you can upload it to the **[KITAB diffviewer](https://kitab-project.org/diffViewer/)** to study the alignments with differences highlighted (choose the 'upload from file' option and upload your tsv).

### 5.4 Visualize one-to-all text reuse data

You can upload a pair of one-to-all text reuse files (`<text_id>_all.csv` and `<text_id>_stats.csv`) to the **[KITAB visualisation app](https://kitab-project.org/explore/#/visualise/)** (choose "Upload a tsv file") to visualize all text reuse in one text detected across the entire corpus.

More info on this visualisation can be found [here](https://kitab-project.org/data/viz#corpus-wide-text-reuse-visualisation-scatter-plot)

# Credits

The idea for this notebook was adapted from a resource created by David Smith (the creator of passim). The original notebook can be found [here](https://github.com/dasmiq/passim/blob/main/docs/passim_quickstart.ipynb)

Scripts for creating passim inputs and processing outputs according to OpenITI schemas were developed by Masoumeh Seydi as part of the KITAB project (they are based on the processing scripts used by the team in their routine passim runs). If these scripts are reused outside of this notebook, it is recommended that you cite Masoumeh (for example, in the code comments or in the doc string of the relevant functions).

The documentation in this notebook was composed jointly by Masoumeh Seydi and Mathew Barber, and lightly edited by Peter Verkinderen.

Thank you in advance for crediting the hard work of the team!